In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool

import subprocess

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import zscore

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.columns = ['chrom', 'start', 'end']
pmds.shape

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic'
comps = pd.read_csv(f'{indir}/csv/compartments/cooltools/liftover/merged.comps.100kb.csv',
                    sep='\t', index_col=0)
comps.index = [f'{x}_{y}' for x,y in zip(comps['chrom'], comps['start'])]
comps.shape

In [ ]:
comps.head()

In [ ]:
bins_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/windows/100kb/GRCh38.100kb.bed'
bins = pd.read_csv(bins_path, sep='\t', header=None)
bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
bins.shape

In [ ]:
s = clusts[0]
start_comps = comps[comps['sample'] == s]
start_comps.shape

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)

### let's start by summarizing one cluster

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_hic/bed/compartments/cooltools/100KB/liftover'
s = 'Start'
tmp_comps = pd.read_csv(f'{indir}/{s}.A.bed', sep='\t', header=None)
tmp_comps.columns = ['chrom', 'start', 'end', 'E1_comp']
comps_bt = BedTool.from_dataframe(tmp_comps)
print(comps_bt.jaccard(pmds_bt)['jaccard'])

### B
tmp_comps = pd.read_csv(f'{indir}/{s}.B.bed', sep='\t', header=None)
tmp_comps.columns = ['chrom', 'start', 'end', 'E1_comp']
comps_bt = BedTool.from_dataframe(tmp_comps)
print(comps_bt.jaccard(pmds_bt)['jaccard'])

### let's take the correlation of Start 100kb windows inside PMDs

In [ ]:
pmds.head()

In [ ]:
size = 1e5

rounded_pmds = pmds.copy()
rounded_pmds['start'] = rounded_pmds['start'].astype(int)
rounded_pmds['end'] = rounded_pmds['end'].astype(int)
rounded_pmds['start'] = [int(x) for x in (rounded_pmds['start'] // size) * size]
rounded_pmds['end'] = [int(x) for x in (rounded_pmds['end'] // size) * size]

mask = (rounded_pmds['start'] == rounded_pmds['end'])
final_pmds = rounded_pmds[~mask]
final_pmds.index = [f'{x}_{y}' for x,y in zip(final_pmds['chrom'], final_pmds['start'])]
final_pmds['size'] = final_pmds['end'] - final_pmds['start']
final_pmds.shape

In [ ]:
final_pmds.head()

In [ ]:
s = 'Start'
comps_df = pd.DataFrame(index=['A', 'B'])

tmp_comps = comps[comps['sample'] == s]

pmds_bt = BedTool.from_dataframe(pmds)
comps_bt = BedTool.from_dataframe(tmp_comps)

comps_inter = comps_bt.intersect(pmds_bt).to_dataframe()
comps_inter.columns = tmp_comps.columns

comps_dist = comps_inter['E1_comp'].value_counts()
comps_df[s] = comps_dist
    
comps_df = comps_df.T
comps_frac = comps_df.div(comps_df.sum(axis=1), axis=0)
comps_frac['sample'] = comps_frac.index
comps_frac

In [ ]:
clust_list = []
for s in tmp_order:
    comps_df = pd.DataFrame(index=['A', 'B'])

    tmp_comps = comps[comps['sample'] == s]

    pmds_bt = BedTool.from_dataframe(pmds)
    comps_bt = BedTool.from_dataframe(tmp_comps)

    comps_inter = comps_bt.intersect(pmds_bt).to_dataframe()
    comps_inter.columns = tmp_comps.columns

    comps_dist = comps_inter['E1_comp'].value_counts()
    comps_df[s] = comps_dist

    comps_df = comps_df.T
    comps_frac = comps_df.div(comps_df.sum(axis=1), axis=0)
    comps_frac['cluster'] = comps_frac.index
    clust_list.append(comps_frac)
    
merged_frac = pd.concat(clust_list)
merged_frac.head()

In [ ]:
to_plot = merged_frac.melt(id_vars=['cluster'])
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(4, 4))

x_lab = 'cluster'
y_lab = 'value'
hue = 'variable'

ax = sns.barplot(data=to_plot, x=x_lab, y=y_lab, hue=hue)
ax.set_xlabel('', fontsize=8)
ax.set_ylabel('', fontsize=8)
ax.set_title('% of 100KB bins in PMDs assigned to A/B compartments', fontsize=12)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)
ax.set_ylim(0, 1.0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/3c_b_comps_pmds_concordance.csv', sep='\t')

### let's do the same thing but get the compartment score distribution

In [ ]:
tmp_samples = clusts[:4]

select = ['E1', 'sample']
tmp_results = []
for s in clusts:
    tmp_comps = comps[comps['sample'] == s]

    pmds_bt = BedTool.from_dataframe(pmds)
    comps_bt = BedTool.from_dataframe(tmp_comps)

    comps_inter = comps_bt.intersect(pmds_bt).to_dataframe()
    comps_inter.columns = tmp_comps.columns
    
    mask = tmp_comps.index.isin(comps_inter.index)
    comps_non_inter = tmp_comps[~mask]
    
    final_inter = comps_inter[select]
    final_inter['type'] = 'pmds'
    
    final_non_inter = comps_non_inter[select]
    final_non_inter['type'] = 'non_pmds'

    tmp_results.append(final_inter)
    tmp_results.append(final_non_inter)

In [ ]:
to_plot = pd.concat(tmp_results)
to_plot['cluster'] = [x.split('_')[0] for x in to_plot['sample']]
to_plot['donor'] = [x.split('_')[1] for x in to_plot['sample']]
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(20, 4))

x_lab = 'sample'
y_lab = 'E1'
hue = 'type'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=8)
ax.set_ylabel('', fontsize=8)
ax.set_title('Compartment score distribution inside/outside of PMDs', fontsize=12)
ax.grid(False)
ax.tick_params(axis='x', labelsize=8, labelrotation=45)
ax.tick_params(axis='y', labelsize=8, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
# Facet across donors
g = sns.catplot(
    data=to_plot,
    x='sample',
    y='E1',
    hue='type',
    col='donor',          # facet across donors
    kind='box',
    sharey=True,
    showfliers=False,
    height=4,
    aspect=1.2,
    col_wrap=2
)

# Adjust labels and titles
g.set_axis_labels("", "E1")
g.set_titles("Donor {col_name}")
g.set_xticklabels(rotation=45, fontsize=8)
g.set(ylim=(to_plot['E1'].min(), to_plot['E1'].max()))
g.add_legend(title='Type')
plt.subplots_adjust(top=0.8)
g.fig.suptitle('Compartment score distribution inside/outside of PMDs')

plt.show()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/pmds_3c_concordance.csv', sep='\t')

In [ ]:
!date